# Dataset Builder Smoke Test

This notebook runs the PGN dataset builder on a tiny synthetic PGN and inspects the generated JSONL shards and manifest. It is for quick local sanity checks, not reportable experiments.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [ ]:
import json
import tempfile
import textwrap

import chess

from mcchess.board import BOARD_TENSOR_SHAPE, encode_board, move_to_index
from mcchess.data import build_dataset

## Tiny PGN

In [ ]:
tiny_pgn = textwrap.dedent(
    """\
    [Event "Fools Mate"]
    [Result "0-1"]

    1. f3 e5 2. g4 Qh4# 0-1

    [Event "Scholars Mate"]
    [Result "1-0"]

    1. e4 e5 2. Qh5 Nc6 3. Bc4 Nf6 4. Qxf7# 1-0

    [Event "Four Ply Draw"]
    [Result "1/2-1/2"]

    1. e4 e5 2. Nf3 Nf6 1/2-1/2

    [Event "Unknown Result"]
    [Result "*"]

    1. e4 e5 *
    """
)

work_handle = tempfile.TemporaryDirectory()
work_dir = Path(work_handle.name)
source_path = work_dir / "tiny.pgn"
output_dir = work_dir / "processed"
manifest_path = work_dir / "manifest.json"

source_path.write_text(tiny_pgn)

{
    "source": str(source_path),
    "output_dir": str(output_dir),
    "manifest": str(manifest_path),
}

## Build Dataset

In [ ]:
build_dataset(
    source_path,
    output_dir,
    manifest_path,
    source_description="synthetic PGN for dataset builder smoke test",
    split_ratios=(0.34, 0.33, 0.33),
    split_seed=14,
    filters={"purpose": "smoke_test"},
)

manifest = json.loads(manifest_path.read_text())
manifest

In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]


samples_by_split = {
    split: read_jsonl(output_dir / f"{split}.jsonl")
    for split in ("train", "val", "test")
}
all_samples = [sample for rows in samples_by_split.values() for sample in rows]

{
    "positions_per_split": {split: len(rows) for split, rows in samples_by_split.items()},
    "game_ids_per_split": {
        split: sorted({row["game_id"] for row in rows})
        for split, rows in samples_by_split.items()
    },
    "total_positions": len(all_samples),
}

## Inspect Samples

In [ ]:
all_samples[:5]

## Contract Checks

In [ ]:
assert manifest["num_games_raw"] == 4
assert manifest["num_games_used"] == 3
assert manifest["num_games_skipped"] == 1
assert manifest["num_positions"] == len(all_samples)
assert manifest["split"]["positions_per_split"] == {
    split: len(rows) for split, rows in samples_by_split.items()
}

game_ids_by_split = {
    split: {row["game_id"] for row in rows}
    for split, rows in samples_by_split.items()
}
assert game_ids_by_split["train"].isdisjoint(game_ids_by_split["val"])
assert game_ids_by_split["train"].isdisjoint(game_ids_by_split["test"])
assert game_ids_by_split["val"].isdisjoint(game_ids_by_split["test"])

"manifest and split checks passed"

In [ ]:
result_value = {"1-0": 1.0, "0-1": -1.0, "1/2-1/2": 0.0}
checked = []

for sample in all_samples:
    board = chess.Board(sample["fen"])
    move = chess.Move.from_uci(sample["move_uci"])
    tensor = encode_board(board)
    expected_value = result_value[sample["result"]] if board.turn == chess.WHITE else -result_value[sample["result"]]

    assert move in board.legal_moves
    assert sample["policy_index"] == move_to_index(board, move)
    assert tuple(tensor.shape) == BOARD_TENSOR_SHAPE
    assert sample["value"] == expected_value

    if len(checked) < 8:
        checked.append(
            {
                "game_id": sample["game_id"],
                "ply": sample["ply"],
                "turn": "white" if board.turn == chess.WHITE else "black",
                "move": sample["move_uci"],
                "policy_index": sample["policy_index"],
                "value": sample["value"],
                "tensor_shape": tuple(tensor.shape),
            }
        )

checked